In [ ]:
!pip install -q transformers accelerate datasets

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/RL/RL_Scene_Graphs
%ls

/content/drive/MyDrive/RL/RL_Scene_Graphs
datasets/  Image_Caption_Data_Prep.ipynb


In [3]:
from datasets import load_dataset

### VG DATA

In [4]:
db_train_vg = load_dataset("JosephZ/vg150_train_sgg_prompt")["train"]
db_val_vg = load_dataset("JosephZ/vg150_val_sgg_prompt")["train"]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [6]:
db_val_vg.save_to_disk("./datasets/vg150_val_sgg_prompt")

Saving the dataset (0/2 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

### PSG DATA

In [9]:
db_val_psg = load_dataset("JosephZ/psg_test_sg")["train"]

In [10]:
# save to disk
db_val_psg.save_to_disk("./datasets/psg_test_sg")

Saving the dataset (0/3 shards):   0%|          | 0/2186 [00:00<?, ? examples/s]

In [11]:
db_val_psg

Dataset({
    features: ['image_id', 'image', 'objects', 'relationships'],
    num_rows: 2186
})

In [13]:
db_train_vg

Dataset({
    features: ['image_id', 'image', 'prompt_open', 'prompt_close', 'objects', 'relationships'],
    num_rows: 56224
})

In [19]:
# check if all prompt_open and prompt_close are the same
ref_prompt_open = db_train_vg[0]["prompt_open"]
ref_prompt_close = db_train_vg[0]["prompt_close"]
cnt = 0
for row in db_train_vg:
  cur_prompt_open = row["prompt_open"]
  cur_prompt_close = row["prompt_close"]

  if cur_prompt_open != ref_prompt_open or cur_prompt_close != ref_prompt_close:
    print("=====DIFFERENT PROMPT=====")
    print(cur_prompt_open)
    print("-------")
    print(cur_prompt_close)
    print("ROW NUMBER ", cnt)
    break

  cnt += 1

=====DIFFERENT PROMPT=====
Generate a structured scene graph for an image of size (662 x 1000) using the following format:

<answer>
{
  "objects": [
    {"id": "object_name.number", "bbox": [x1, y1, x2, y2]},
    ...
  ],
  "relationships": [
    {"subject": "object_name.number", "predicate": "relationship_type", "object": "object_name.number"},
    ...
  ]
}
</answer>

### **Guidelines:**
- **Objects:**
  - Assign a unique ID for each object using the format `"object_name.number"` (e.g., `"person.1"`, `"bike.2"`).
  - Provide its bounding box `[x1, y1, x2, y2]` in integer pixel format.
  - Include all visible objects, even if they have no relationships.

- **Relationships:**
  - Represent interactions accurately using `"subject"`, `"predicate"`, and `"object"`.
  - Omit relationships for orphan objects.

### **Example Output:**
<answer>
{
  "objects": [
    {"id": "person.1", "bbox": [120, 200, 350, 700]},
    {"id": "bike.2", "bbox": [100, 600, 400, 800]},
    {"id": "helmet.3", "bb

In [20]:
diff_prompt_open = db_train_vg[1]["prompt_open"]
diff_prompt_close = db_train_vg[1]["prompt_close"]
print(diff_prompt_open)

Generate a structured scene graph for an image of size (662 x 1000) using the following format:

<answer>
{
  "objects": [
    {"id": "object_name.number", "bbox": [x1, y1, x2, y2]},
    ...
  ],
  "relationships": [
    {"subject": "object_name.number", "predicate": "relationship_type", "object": "object_name.number"},
    ...
  ]
}
</answer>

### **Guidelines:**
- **Objects:**
  - Assign a unique ID for each object using the format `"object_name.number"` (e.g., `"person.1"`, `"bike.2"`).
  - Provide its bounding box `[x1, y1, x2, y2]` in integer pixel format.
  - Include all visible objects, even if they have no relationships.

- **Relationships:**
  - Represent interactions accurately using `"subject"`, `"predicate"`, and `"object"`.
  - Omit relationships for orphan objects.

### **Example Output:**
<answer>
{
  "objects": [
    {"id": "person.1", "bbox": [120, 200, 350, 700]},
    {"id": "bike.2", "bbox": [100, 600, 400, 800]},
    {"id": "helmet.3", "bbox": [150, 150, 280, 240]},

In [21]:
print(ref_prompt_open)

Generate a structured scene graph for an image of size (1024 x 768) using the following format:

<answer>
{
  "objects": [
    {"id": "object_name.number", "bbox": [x1, y1, x2, y2]},
    ...
  ],
  "relationships": [
    {"subject": "object_name.number", "predicate": "relationship_type", "object": "object_name.number"},
    ...
  ]
}
</answer>

### **Guidelines:**
- **Objects:**
  - Assign a unique ID for each object using the format `"object_name.number"` (e.g., `"person.1"`, `"bike.2"`).
  - Provide its bounding box `[x1, y1, x2, y2]` in integer pixel format.
  - Include all visible objects, even if they have no relationships.

- **Relationships:**
  - Represent interactions accurately using `"subject"`, `"predicate"`, and `"object"`.
  - Omit relationships for orphan objects.

### **Example Output:**
<answer>
{
  "objects": [
    {"id": "person.1", "bbox": [120, 200, 350, 700]},
    {"id": "bike.2", "bbox": [100, 600, 400, 800]},
    {"id": "helmet.3", "bbox": [150, 150, 280, 240]},

In [22]:
# save ref_prompt_open and close in a txt file
with open("./datasets/ref_prompt_open.txt", "w") as f:
  f.write(ref_prompt_open)

with open("./datasets/ref_prompt_close.txt", "w") as f:
  f.write(ref_prompt_close)

In [25]:
db_train_psg = load_dataset("JosephZ/psg_train_sg", split="train", streaming=True)

Resolving data files:   0%|          | 0/46 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/46 [00:00<?, ?it/s]

In [30]:
db_train_psg_subset = list(db_train_psg.take(10))
db_train_psg_subset

[{'image_id': '107902',
  'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=480x640>,
  'objects': '[{"id": "person.1", "bbox": [49, 162, 207, 617]}, {"id": "person.2", "bbox": [259, 158, 397, 631]}, {"id": "umbrella.3", "bbox": [157, 75, 469, 279]}, {"id": "tree-merged.4", "bbox": [0, 0, 480, 131]}, {"id": "fence-merged.5", "bbox": [0, 112, 480, 323]}, {"id": "sky-other-merged.6", "bbox": [215, 0, 480, 54]}, {"id": "grass-merged.7", "bbox": [0, 285, 480, 640]}]',
  'relationships': '[{"subject": "person.1", "predicate": "beside", "object": "person.2"}, {"subject": "person.1", "predicate": "standing on", "object": "grass-merged.7"}, {"subject": "person.2", "predicate": "holding", "object": "umbrella.3"}, {"subject": "person.2", "predicate": "standing on", "object": "grass-merged.7"}, {"subject": "fence-merged.5", "predicate": "in front of", "object": "tree-merged.4"}, {"subject": "sky-other-merged.6", "predicate": "over", "object": "tree-merged.4"}]'},
 {'image_id': '107905

In [32]:
for row in db_train_psg_subset[:1]:
  print(row.keys())

dict_keys(['image_id', 'image', 'objects', 'relationships'])
